# ML model results

Reads **all experimental runs** from `saved/ml_results.parquet` (written by `02_ml_models_fit.ipynb`).

Six models × four feature tables:

| Dataset | Features |
|---|---|
| Baseline | original columns, no `uid` / `uid2` / `DT_*` |
| Feature Engineering | plus `uid`, `uid2`, `DT_*` |
| Reduced Baseline | Table 3 filters, no `uid` / `uid2` / `DT_*` |
| Reduced Feature Engineering | reduced plus surviving `uid` / `uid2` / `DT_*` |

Models: logistic regression, decision tree, random forest, LightGBM, XGBoost, CatBoost.


In [1]:
import warnings
import numpy as np
import pandas as pd
import json
import joblib
from pathlib import Path
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    balanced_accuracy_score,
    matthews_corrcoef,
    classification_report,
)

warnings.filterwarnings("ignore")
pd.options.display.precision = 4

print("ML model results")

ML model results


In [2]:
# Environment & Paths Setup

try:
    import google.colab
    IS_COLAB = True
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    ROOT = Path("/content/drive/MyDrive/minor-thesis")
else:
    ROOT = Path.cwd()

DATASET_PATH = ROOT / "dataset"
SAVED_PATH = ROOT / "saved"
MODEL_DIR = SAVED_PATH / "optimized_models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print(f'Environment: {"Google Colab" if IS_COLAB else "Local"}')
print(f"Dataset path: {DATASET_PATH}")
print(f"Results: {SAVED_PATH / 'ml_results.parquet'}")
print(f"Model dir: {MODEL_DIR}")

Environment: Local
Dataset path: d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\dataset
Results: d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\saved\ml_results.parquet
Model dir: d:\source\RMIT\master-of-ai-new\2026-semester-02\minor-thesis\saved\optimized_models


## Experiment results

One table of every trained run in `saved/ml_results.parquet` (6 models × 4 datasets). Re-run any missing family in `02_ml_models_fit.ipynb`.


In [3]:
from IPython.display import display

ml_results_path = SAVED_PATH / "ml_results.parquet"
all_results = pd.read_parquet(ml_results_path)
all_results = all_results[
    ~all_results["Model"].astype(str).str.startswith(("SVM -", "KNN -"))
]

MODEL_ORDER = [
    "LogisticRegression",
    "DecisionTree",
    "RandomForest",
    "LightGBM",
    "XGBoost",
    "CatBoost",
]
DATASET_ORDER = [
    "Baseline",
    "Feature Engineering",
    "Reduced Baseline",
    "Reduced Feature Engineering",
]
NAME_PREFIX = {
    "LogisticRegression": "Logistic Regression",
    "DecisionTree": "Decision Tree",
    "RandomForest": "RF",
    "LightGBM": "LightGBM",
    "XGBoost": "XGBoost",
    "CatBoost": "CatBoost",
}

if "Dataset" not in all_results.columns:
    all_results["Dataset"] = all_results["Model"].astype(str).str.split(" - ", n=1).str[1]

expected = {
    f"{prefix} - {dataset}"
    for prefix in NAME_PREFIX.values()
    for dataset in DATASET_ORDER
}
missing = sorted(expected - set(all_results["Model"].astype(str)))

show_cols = [
    "ModelType",
    "Dataset",
    "Features",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "ROC-AUC",
    "TuneROC-AUC",
    "PR-AUC",
    "Balanced Accuracy",
    "MCC",
    "TP",
    "FP",
    "FN",
    "TN",
    "BestParams",
]
show_cols = [c for c in show_cols if c in all_results.columns]

results_table = all_results.copy()
results_table["ModelType"] = pd.Categorical(
    results_table["ModelType"], categories=MODEL_ORDER, ordered=True
)
results_table["Dataset"] = pd.Categorical(
    results_table["Dataset"], categories=DATASET_ORDER, ordered=True
)
results_table = (
    results_table.sort_values(["ModelType", "Dataset"])[show_cols].reset_index(drop=True)
)

print(
    f"{ml_results_path.name}: {len(results_table)} trained runs "
    f"(expected 24 = 6 models x 4 datasets)"
)
if missing:
    print("Not trained yet:")
    for name in missing:
        print(f"  {name}")
print()
display(results_table)


ml_results.parquet: 25 trained runs (expected 32 = 8 models x 4 datasets)
Not trained yet:
  KNN - Baseline
  KNN - Feature Engineering
  KNN - Reduced Baseline
  KNN - Reduced Feature Engineering
  SVM - Feature Engineering
  SVM - Reduced Baseline
  SVM - Reduced Feature Engineering



,ModelType,Dataset,Features,Accuracy,Precision,Recall,F1,ROC-AUC,PR-AUC,Balanced Accuracy,MCC,TP,FP,FN,TN
0,LogisticRegression,Baseline,432,0.7034,0.0870,0.8024,0.1569,0.8324,0.1987,0.7511,0.1967,3261,34230,803,79814
1,LogisticRegression,Feature Engineering,439,0.7020,0.0866,0.8024,0.1563,0.8315,0.1971,0.7504,0.1959,3261,34397,803,79647
2,LogisticRegression,Reduced Baseline,342,0.6905,0.0839,0.8056,0.1519,0.8270,0.1868,0.7460,0.1907,3274,35761,790,78283
3,LogisticRegression,Reduced Feature Engineering,348,0.6954,0.0847,0.8012,0.1533,0.8268,0.1866,0.7464,0.1918,3256,35165,808,78879
4,DecisionTree,Baseline,432,0.9143,0.2104,0.5408,0.3029,0.6859,0.2533,0.7343,0.3007,2198,8250,1866,105794
5,DecisionTree,Feature Engineering,439,0.9138,0.2075,0.5337,0.2988,0.6714,0.2451,0.7305,0.2959,2169,8285,1895,105759
6,DecisionTree,Reduced Baseline,342,0.9038,0.1883,0.5421,0.2795,0.6818,0.2143,0.7294,0.2799,2203,9498,1861,104546
7,DecisionTree,Reduced Feature Engineering,348,0.9055,0.1903,0.5372,0.2811,0.6764,0.2247,0.7279,0.2805,2183,9286,1881,104758
8,RandomForest,Baseline,432,0.9742,0.8010,0.3317,0.4691,0.9052,0.5267,0.6644,0.5056,1348,335,2716,113709
9,RandomForest,Feature Engineering,439,0.9742,0.8149,0.3238,0.4635,0.9074,0.5341,0.6606,0.5041,1316,299,2748,113745


## Top 20 feature importances

Each training run stores its top 20 features (with `%` of total model importance) in `Top20Importances`. Re-run those experiments in `02_ml_models_fit.ipynb` if the column is missing.


In [4]:
def parse_top20(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        return pd.DataFrame(json.loads(text))
    if isinstance(value, (list, tuple)):
        return pd.DataFrame(value)
    return None


def show_top20(model_name):
    rows = all_results[all_results["Model"] == model_name]
    if rows.empty:
        print(f"{model_name}: not in ml_results.parquet")
        return
    row = rows.iloc[0]
    if "Top20Importances" not in all_results.columns:
        print("Top20Importances column missing — re-run 02_ml_models.ipynb")
        return
    table = parse_top20(row["Top20Importances"])
    if table is None or table.empty:
        print(f"{model_name}: no top-20 saved yet — re-run that experiment in 02_ml_models.ipynb")
        return
    table = table.rename(
        columns={
            "rank": "Rank",
            "feature": "Feature",
            "importance": "Importance",
            "importance_pct": "Importance %",
        }
    )
    print(
        f"\n{model_name}  |  {int(row['Features'])} features  |  "
        f"ROC-AUC {row['ROC-AUC']:.4f}  |  F1 {row['F1']:.4f}"
    )
    display(table)


if "Top20Importances" not in all_results.columns:
    print("Top20Importances is not in ml_results.parquet yet.")
    print("Re-run the experiment cells in 02_ml_models.ipynb, then re-load this notebook.")
else:
    n_saved = all_results["Top20Importances"].notna().sum()
    print(f"Runs with top-20 importances: {n_saved} / {len(all_results)}")

    highlight = [
        "Logistic Regression - Baseline",
        "Decision Tree - Baseline",
        "RF - Baseline",
        "LightGBM - Baseline",
        "XGBoost - Baseline",
        "CatBoost - Baseline",
        "RF - Feature Engineering",
        "LightGBM - Feature Engineering",
        "XGBoost - Feature Engineering",
        "CatBoost - Feature Engineering",
        "RF - Reduced Baseline",
        "LightGBM - Reduced Baseline",
        "XGBoost - Reduced Baseline",
        "CatBoost - Reduced Baseline",
        "RF - Reduced Feature Engineering",
        "LightGBM - Reduced Feature Engineering",
        "XGBoost - Reduced Feature Engineering",
        "CatBoost - Reduced Feature Engineering",
    ]
    for name in highlight:
        show_top20(name)

    print("\n===== BEST ROC-AUC PER MODEL FAMILY =====")
    for family in MODEL_ORDER:
        frame = all_results[all_results["ModelType"] == family]
        if frame.empty:
            continue
        best_name = frame.sort_values("ROC-AUC", ascending=False).iloc[0]["Model"]
        print(f"\n{family} best: {best_name}")
        show_top20(best_name)


Runs with top-20 importances: 25 / 25

Logistic Regression - Baseline  |  432 features  |  ROC-AUC 0.8324  |  F1 0.1569


,Rank,Feature,Importance,Importance %
0,1,C11,8.6684,3.2088
1,2,V283,8.4567,3.1304
2,3,C14,8.2745,3.0630
3,4,V300,4.5343,1.6785
4,5,C12,3.8769,1.4351
5,6,V123,3.7938,1.4044
6,7,C4,3.7278,1.3799
7,8,V271,3.5034,1.2968
8,9,V218,2.9186,1.0804
9,10,V105,2.7579,1.0209



Decision Tree - Baseline  |  432 features  |  ROC-AUC 0.6859  |  F1 0.3029


,Rank,Feature,Importance,Importance %
0,1,V258,0.1634,16.3376
1,2,V294,0.0981,9.8128
2,3,C14,0.0781,7.8113
3,4,C8,0.0464,4.6403
4,5,M4,0.0322,3.2170
5,6,TransactionDT,0.0319,3.1946
6,7,card1,0.0315,3.1544
7,8,TransactionAmt,0.0284,2.8444
8,9,card6,0.0241,2.4113
9,10,card2,0.0217,2.1747


RF - Baseline: no top-20 saved yet — re-run that experiment in 02_ml_models.ipynb

SVM - Baseline  |  432 features  |  ROC-AUC 0.8278  |  F1 0.1618


...

,Rank,Feature,Importance,Importance %
0,1,card1,943.0,6.2867
1,2,card2,764.0,5.0933
2,3,TransactionDT,633.0,4.2200
3,4,addr1,604.0,4.0267
4,5,TransactionAmt,576.0,3.8400
5,6,C1,445.0,2.9667
6,7,C13,445.0,2.9667
7,8,D15,396.0,2.6400
8,9,D2,342.0,2.2800
9,10,P_emaildomain,324.0,2.1600



XGBoost best: XGBoost - Reduced Feature Engineering

XGBoost - Reduced Feature Engineering  |  348 features  |  ROC-AUC 0.9034  |  F1 0.3557


,Rank,Feature,Importance,Importance %
0,1,V258,0.1834,18.3425
1,2,V70,0.0801,8.0063
2,3,V91,0.0540,5.4026
3,4,V201,0.0440,4.3986
4,5,V317,0.0356,3.5626
5,6,C4,0.0327,3.2693
6,7,V308,0.0164,1.6382
7,8,V187,0.0160,1.5969
8,9,C14,0.0153,1.5285
9,10,addr2,0.0152,1.5178



CatBoost best: CatBoost - Reduced Feature Engineering

CatBoost - Reduced Feature Engineering  |  348 features  |  ROC-AUC 0.9053  |  F1 0.3055


,Rank,Feature,Importance,Importance %
0,1,C1,11.3063,11.3063
1,2,C14,4.5230,4.5230
2,3,C13,3.9987,3.9987
3,4,card2,3.9457,3.9457
4,5,uid,3.2429,3.2429
5,6,TransactionAmt,2.8581,2.8581
6,7,addr1,2.5968,2.5968
7,8,card6,2.4833,2.4833
8,9,D2,2.1736,2.1736
9,10,R_emaildomain,1.9717,1.9717


## Imbalanced-data read

Accuracy is misleading at a 3.5% fraud rate. Rank the four-dataset runs by ROC-AUC, PR-AUC, F1, and recall.


In [5]:
print("=" * 130)
print("IMBALANCED DATASET ANALYSIS - FOR FRAUD DETECTION")
print("Focus: ROC-AUC, PR-AUC, F1, Recall (NOT Accuracy - accuracy is misleading!)")
print("=" * 130)

print("\n1. KEY METRICS FOR IMBALANCED DATA")
print("-" * 130)
print("""
For fraud detection (imbalanced dataset):

ROC-AUC (0.85-0.95 good)
  - Threshold-independent, handles class imbalance naturally
  - Main metric for comparing models

PR-AUC (Precision-Recall)
  - Especially important when fraud is rare
  - Shows trade-off: catch more frauds vs avoid false alerts

F1 Score
  - Harmonic mean of Precision & Recall
  - Balances both metrics

Recall (Sensitivity)
  - % of actual frauds caught
  - Missing a fraud = business loss

Precision
  - % of fraud alerts that are actually fraud
  - False alert = customer friction

Confusion Matrix (TN, FP, FN, TP)
  - FN (False Negatives) = frauds we missed - WORST outcome
  - FP (False Positives) = customers falsely flagged - customer friction
  - TP (True Positives) = frauds caught - GOOD
  - TN (True Negatives) = legitimate txns correctly allowed - GOOD

Accuracy - MISLEADING for imbalanced data!
  Example: 99% legitimates, 1% fraud
  Model that predicts "always legitimate" = 99% accuracy but CATCHES ZERO FRAUDS
""")

print("\n\n2. TOP CANDIDATES FOR IMBALANCED FRAUD DETECTION (ROC-AUC > 0.90)")
print("-" * 130)

high_roc = all_results[all_results["ROC-AUC"] > 0.90].sort_values(
    ["Features", "ROC-AUC"], ascending=[True, False]
)
display_cols = ["Model", "Features", "ROC-AUC", "PR-AUC", "F1", "Recall", "Precision", "TP", "FP", "FN"]
print(high_roc[display_cols].head(15).to_string(index=False))

print("\n\n3. DETAILED COMPARISON - THREE MAIN CANDIDATES")
print("=" * 130)

focus = [
    "RF - Baseline",
    "RF - Feature Engineering",
    "RF - Reduced Baseline",
    "RF - Reduced Feature Engineering",
    "LightGBM - Reduced Feature Engineering",
    "XGBoost - Reduced Feature Engineering",
    "CatBoost - Reduced Feature Engineering",
]

for model_name in focus:
    row = all_results[all_results["Model"] == model_name]
    if len(row) == 0:
        continue
    row = row.iloc[0]

    print(f"\n{model_name}")
    print("-" * 130)

    print("\nCore Metrics (for imbalanced data):")
    print(f"  ROC-AUC:              {row['ROC-AUC']:.4f}  <- Main comparison metric")
    print(f"  PR-AUC:               {row['PR-AUC']:.4f}  <- Critical for fraud (rare events)")
    print(f"  F1 Score:             {row['F1']:.4f}   <- Balance precision & recall")

    print("\nFraud Detection Performance:")
    print(f"  Recall (catch rate):  {row['Recall']:.4f}   <- % of frauds actually caught")
    print(f"  Precision:            {row['Precision']:.4f}  <- % of alerts that are real frauds")

    print("\nConfusion Matrix Breakdown:")
    tn, fp, fn, tp = int(row["TN"]), int(row["FP"]), int(row["FN"]), int(row["TP"])
    total = tn + fp + fn + tp
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0

    print(f"  True Negatives (TN):   {tn:8d}  <- Legitimate txns correctly allowed")
    print(f"  False Positives (FP):  {fp:8d}  <- Legitimate txns falsely flagged (customer friction)")
    print(f"  False Negatives (FN):  {fn:8d}  <- Frauds missed (WORST - direct loss!)")
    print(f"  True Positives (TP):   {tp:8d}  <- Frauds caught (BEST)")
    print("  -----------------------------------")
    print(f"  Total samples:         {total:8d}")

    print("\nDerived Metrics:")
    print(f"  Specificity:          {specificity:.4f}   <- % of legitimate txns correctly allowed")
    print(f"  Sensitivity:          {sensitivity:.4f}   <- % of frauds caught (same as Recall)")
    print(f"  False Alarm Rate:     {fp / (fp + tn):.4f}   <- % of legitimate txns falsely flagged")
    print(f"  False Negative Rate:  {fn / (fn + tp):.4f}   <- % of frauds missed (minimize this!)")

    print(f"  Features:             {int(row['Features'])} columns")

print("\n\n4. RECOMMENDATION FOR IMBALANCED FRAUD DETECTION")
print("=" * 130)
print("""
KEY INSIGHT FOR IMBALANCED DATA:

Priority:
1. MINIMIZE FN (False Negatives / Missed Frauds) <- Direct business loss
2. MAINTAIN ROC-AUC > 0.90 <- Threshold-independent quality
3. MAXIMIZE F1 & Recall <- Better fraud detection
4. MANAGE FP (False Positives) <- Customer experience
5. IGNORE Accuracy <- Misleading for imbalanced data

BEST CHOICE: re-run 02_ml_models_fit.ipynb, then pick the row with highest PR-AUC / F1 among the four tables.

Compare Baseline vs Feature Engineering vs Reduced Baseline vs Reduced Feature Engineering
for each model family. Group-ablation permutations (Remove C / D / M / id / V) are no longer trained.
""")
print("=" * 130)


IMBALANCED DATASET ANALYSIS - FOR FRAUD DETECTION
Focus: ROC-AUC, PR-AUC, F1, Recall (NOT Accuracy - accuracy is misleading!)

1. KEY METRICS FOR IMBALANCED DATA
----------------------------------------------------------------------------------------------------------------------------------

For fraud detection (imbalanced dataset):

ROC-AUC (0.85-0.95 good)
  - Threshold-independent, handles class imbalance naturally
  - Main metric for comparing models

PR-AUC (Precision-Recall)
  - Especially important when fraud is rare
  - Shows trade-off: catch more frauds vs avoid false alerts

F1 Score
  - Harmonic mean of Precision & Recall
  - Balances both metrics

Recall (Sensitivity)
  - % of actual frauds caught
  - Missing a fraud = business loss

Precision
  - % of fraud alerts that are actually fraud
  - False alert = customer friction

Confusion Matrix (TN, FP, FN, TP)
  - FN (False Negatives) = frauds we missed - WORST outcome
  - FP (False Positives) = customers falsely flagged - c

## Feature-count trade-off

Best model at each width, and the smallest feature set that still clears ROC-AUC 0.90.

In [6]:
print("=" * 100)
print("ANALYZING ALL CONFIGURATIONS: Looking for Best Trade-offs")
print("=" * 100)

print("\n1. BY FEATURE COUNT (Least Parameters)")
print("-" * 100)

for feature_count in sorted(all_results["Features"].unique()):
    group = all_results[all_results["Features"] == feature_count].sort_values(
        "ROC-AUC", ascending=False
    )
    if len(group) > 0:
        best = group.iloc[0]
        print(
            f"\nFeatures: {int(feature_count):3d} | Best: {best['Model'][:40]:40s} | "
            f"ROC-AUC: {best['ROC-AUC']:.4f} | F1: {best['F1']:.4f}"
        )

print("\n\n2. TRADE-OFF ANALYSIS (Best Metrics vs Fewest Features)")
print("-" * 100)

sorted_by_features = all_results.sort_values(["Features", "ROC-AUC"], ascending=[True, False])

print("\nTop performer in each feature-reduction tier:")
seen_features = set()
count = 0
for _, row in sorted_by_features.iterrows():
    if row["Features"] not in seen_features and count < 8:
        seen_features.add(row["Features"])
        print(
            f"  {int(row['Features']):3d} features | {row['Model'][:45]:45s} | "
            f"ROC-AUC: {row['ROC-AUC']:.4f} | F1: {row['F1']:.4f} | Type: {row['ModelType']}"
        )
        count += 1

print("\n\n3. PER MODEL FAMILY (fewest features first)")
print("-" * 100)
for family in MODEL_ORDER:
    frame = all_results[all_results["ModelType"] == family]
    if frame.empty:
        continue
    print(f"\n{family}")
    display(
        frame.nsmallest(10, "Features")[
            [c for c in ["Model", "Features", "ROC-AUC", "PR-AUC", "F1", "Accuracy"] if c in frame.columns]
        ]
    )

print("\n\n4. BEST BY DIFFERENT METRICS")
print("-" * 100)

few = all_results[all_results["Features"] < 100]
if few.empty:
    print("No runs with <100 features (group ablations were removed). Ranking the four tables instead.")
    few = all_results
best_roc_few = few.nlargest(1, "ROC-AUC").iloc[0]
print("\nBest ROC-AUC:")
print(
    f"  {best_roc_few['Model']} | Features: {best_roc_few['Features']:.0f} | "
    f"ROC-AUC: {best_roc_few['ROC-AUC']:.4f} | F1: {best_roc_few['F1']:.4f}"
)
best_f1_few = few.nlargest(1, "F1").iloc[0]
print("\nBest F1:")
print(
    f"  {best_f1_few['Model']} | Features: {best_f1_few['Features']:.0f} | "
    f"ROC-AUC: {best_f1_few['ROC-AUC']:.4f} | F1: {best_f1_few['F1']:.4f}"
)

ranked = all_results.copy()
ranked["Balance"] = (ranked["ROC-AUC"] + ranked["F1"]) / 2
best_balanced_few = ranked.nlargest(1, "Balance").iloc[0]
print("\nBest Balance (ROC-AUC + F1 avg):")
print(
    f"  {best_balanced_few['Model']} | Features: {best_balanced_few['Features']:.0f} | "
    f"ROC-AUC: {best_balanced_few['ROC-AUC']:.4f} | F1: {best_balanced_few['F1']:.4f}"
)

qualifying = all_results[all_results["ROC-AUC"] > 0.90]
if len(qualifying) > 0:
    best_minimal = qualifying.nsmallest(1, "Features").iloc[0]
    print("\nMinimum features with ROC-AUC >0.90:")
    print(
        f"  {best_minimal['Model']} | Features: {best_minimal['Features']:.0f} | "
        f"ROC-AUC: {best_minimal['ROC-AUC']:.4f} | F1: {best_minimal['F1']:.4f}"
    )

print("\n\n" + "=" * 100)


ANALYZING ALL CONFIGURATIONS: Looking for Best Trade-offs

1. BY FEATURE COUNT (Least Parameters)
----------------------------------------------------------------------------------------------------

Features: 342 | Best: LightGBM - Reduced Baseline              | ROC-AUC: 0.9061 | F1: 0.3269

Features: 348 | Best: RF - Reduced Feature Engineering         | ROC-AUC: 0.9066 | F1: 0.4645

Features: 432 | Best: RF - Baseline                            | ROC-AUC: 0.9052 | F1: 0.4691

Features: 439 | Best: RF - Feature Engineering                 | ROC-AUC: 0.9074 | F1: 0.4635


2. TRADE-OFF ANALYSIS (Best Metrics vs Fewest Features)
----------------------------------------------------------------------------------------------------

Top performer in each feature-reduction tier:
  342 features | LightGBM - Reduced Baseline                   | ROC-AUC: 0.9061 | F1: 0.3269 | Type: LightGBM
  348 features | RF - Reduced Feature Engineering              | ROC-AUC: 0.9066 | F1: 0.4645 | Type: Ra

,Model,Features,ROC-AUC,PR-AUC,F1,Accuracy
3,Logistic Regression - Reduced Baseline,342,0.8270,0.1868,0.1519,0.6905
4,Logistic Regression - Reduced Feature Engineering,348,0.8268,0.1866,0.1533,0.6954
1,Logistic Regression - Baseline,432,0.8324,0.1987,0.1569,0.7034
2,Logistic Regression - Feature Engineering,439,0.8315,0.1971,0.1563,0.7020



DecisionTree


,Model,Features,ROC-AUC,PR-AUC,F1,Accuracy
7,Decision Tree - Reduced Baseline,342,0.6818,0.2143,0.2795,0.9038
8,Decision Tree - Reduced Feature Engineering,348,0.6764,0.2247,0.2811,0.9055
5,Decision Tree - Baseline,432,0.6859,0.2533,0.3029,0.9143
6,Decision Tree - Feature Engineering,439,0.6714,0.2451,0.2988,0.9138



RandomForest


...


XGBoost


,Model,Features,ROC-AUC,PR-AUC,F1,Accuracy
19,XGBoost - Reduced Baseline,342,0.9016,0.5177,0.3537,0.9111
20,XGBoost - Reduced Feature Engineering,348,0.9034,0.5197,0.3557,0.9108
17,XGBoost - Baseline,432,0.9022,0.5176,0.3592,0.9125
18,XGBoost - Feature Engineering,439,0.9027,0.5217,0.3580,0.9126



CatBoost


,Model,Features,ROC-AUC,PR-AUC,F1,Accuracy
23,CatBoost - Reduced Baseline,342,0.9043,0.4855,0.3021,0.8806
24,CatBoost - Reduced Feature Engineering,348,0.9053,0.4892,0.3055,0.8823
21,CatBoost - Baseline,432,0.9034,0.4971,0.3081,0.8838
22,CatBoost - Feature Engineering,439,0.9048,0.4992,0.3065,0.8823




4. BEST BY DIFFERENT METRICS
----------------------------------------------------------------------------------------------------
No runs with <100 features (group ablations were removed). Ranking the four tables instead.

Best ROC-AUC:
  RF - Feature Engineering | Features: 439 | ROC-AUC: 0.9074 | F1: 0.4635

Best F1:
  RF - Reduced Baseline | Features: 342 | ROC-AUC: 0.9050 | F1: 0.4720

Best Balance (ROC-AUC + F1 avg):
  RF - Reduced Baseline | Features: 342 | ROC-AUC: 0.9050 | F1: 0.4720

Minimum features with ROC-AUC >0.90:
  RF - Reduced Baseline | Features: 342 | ROC-AUC: 0.9050 | F1: 0.4720




## Compact Random Forest refit

Train an optimized RF on the **reduced feature-engineering** table from `01_clean_dataset.ipynb` (Table 3 filters, including `uid` / `uid2`).


In [7]:
# Load Data (for the compact refit after ranking ml_results.parquet)

# Load Data

train = pd.read_parquet(DATASET_PATH / "merged_train_reduced.parquet")
test = pd.read_parquet(f'{DATASET_PATH}/merged_test.parquet')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f'\nMissing values in train:')
print(train.isnull().sum()[train.isnull().sum() > 0])


Train shape: (590540, 350)
Test shape: (506691, 440)

Missing values in train:
Series([], dtype: int64)


In [8]:
# Feature set: reduced table with uid / uid2 (Reduced Feature Engineering)
train = pd.read_parquet(DATASET_PATH / "merged_train_reduced.parquet")
train_sorted = train.sort_values("TransactionDT").reset_index(drop=True)
optimal_cols = [c for c in train_sorted.columns if c not in ["isFraud", "TransactionID"]]
y = train_sorted["isFraud"]
X = train_sorted[optimal_cols]
print("Optimal Feature Set: reduced feature engineering")
print(f"  Total features: {len(optimal_cols)}")
print(f"  Table: {train_sorted.shape}")


Optimal Feature Set: reduced feature engineering
  Total features: 348
  Table: (590540, 350)


In [9]:
# Data Split (80/20 temporal)

split_idx = int(len(X) * 0.8)

X_train = X.iloc[:split_idx]
X_valid = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_valid = y.iloc[split_idx:]

print(f'Train/Valid Split (Temporal 80/20):')
print(f'  Train: {len(X_train):,} samples')
print(f'  Valid: {len(X_valid):,} samples')
print(f'  Fraud rate (train): {y_train.mean():.4f}')
print(f'  Fraud rate (valid): {y_valid.mean():.4f}')

Train/Valid Split (Temporal 80/20):
  Train: 472,432 samples
  Valid: 118,108 samples
  Fraud rate (train): 0.0351
  Fraud rate (valid): 0.0344


In [10]:
# Baseline from the experiment table
baseline_result = all_results[all_results["Model"] == "RF - Baseline"].iloc[0]

print("Baseline Model (Full Features 437):")
print("  Model: Random Forest (n_estimators=300)")
print(f"  ROC-AUC: {baseline_result['ROC-AUC']:.4f}")
print(f"  PR-AUC: {baseline_result['PR-AUC']:.4f}")
print(f"  F1: {baseline_result['F1']:.4f}")
print(f"  Accuracy: {baseline_result['Accuracy']:.4f}")

Baseline Model (Full Features 437):
  Model: Random Forest (n_estimators=300)
  ROC-AUC: 0.9052
  PR-AUC: 0.5267
  F1: 0.4691
  Accuracy: 0.9742


In [ ]:
# Hyperparameter Tuning via GridSearchCV
# 
# Optimize n_estimators, max_depth, and min_samples_split
# for the reduced feature set

param_grid = {
    'n_estimators': [100, 150, 200],  # Reduce from baseline 300
    'max_depth': [None, 15, 20],
    'min_samples_split': [5, 10],
}

rf_base = RandomForestClassifier(
    class_weight='balanced',
    random_state=RANDOM_SEED,
    n_jobs=-1
)

print('GridSearchCV Configuration:')
print(f'  Parameters to search: {param_grid}')
print(f'  CV folds: 3')
print(f'  Scoring: roc_auc')
print(f'  Running grid search...')

grid_search = GridSearchCV(
    rf_base,
    param_grid,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f'\nBest Parameters:')
for key, value in grid_search.best_params_.items():
    print(f'  {key}: {value}')
print(f'Best CV ROC-AUC: {grid_search.best_score_:.4f}')

GridSearchCV Configuration:
  Parameters to search: {'n_estimators': [100, 150, 200], 'max_depth': [None, 15, 20], 'min_samples_split': [5, 10]}
  CV folds: 3
  Scoring: roc_auc
  Running grid search...
Fitting 3 folds for each of 18 candidates, totalling 54 fits


In [ ]:
# Get Best Model

best_model = grid_search.best_estimator_

print(f'Optimized Model Configuration:')
print(f'  n_estimators: {best_model.n_estimators}')
print(f'  max_depth: {best_model.max_depth}')
print(f'  min_samples_split: {best_model.min_samples_split}')
print(f'  class_weight: balanced')
print(f'  Features: {len(optimal_cols)}')

In [ ]:
# Evaluate Optimized Model

y_pred = best_model.predict(X_valid)
y_pred_prob = best_model.predict_proba(X_valid)[:, 1]

accuracy = accuracy_score(y_valid, y_pred)
precision = precision_score(y_valid, y_pred, zero_division=0)
recall = recall_score(y_valid, y_pred, zero_division=0)
f1 = f1_score(y_valid, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_valid, y_pred_prob)
pr_auc = average_precision_score(y_valid, y_pred_prob)
balanced_acc = balanced_accuracy_score(y_valid, y_pred)
mcc = matthews_corrcoef(y_valid, y_pred)
cm = confusion_matrix(y_valid, y_pred)

print('Optimized Model Metrics:')
print(f'  Accuracy: {accuracy:.4f}')
print(f'  Precision: {precision:.4f}')
print(f'  Recall: {recall:.4f}')
print(f'  F1 Score: {f1:.4f}')
print(f'  ROC-AUC: {roc_auc:.4f}')
print(f'  PR-AUC: {pr_auc:.4f}')
print(f'  Balanced Accuracy: {balanced_acc:.4f}')
print(f'  MCC: {mcc:.4f}')
print(f'\nConfusion Matrix [TN, FP, FN, TP]:')
print(f'  [[{cm[0,0]}, {cm[0,1]}], [{cm[1,0]}, {cm[1,1]}]]')

In [ ]:
# Classification Report

print('\nClassification Report:')
print(classification_report(
    y_valid, y_pred,
    target_names=['Legitimate', 'Fraud'],
    digits=4,
    zero_division=0
))

In [ ]:
# Model Comparison Table

comparison = pd.DataFrame({
    'Model': ['Baseline (Full Features)', 'Optimized (60 Features)'],
    'Features': [int(baseline_result['Features']), len(optimal_cols)],
    'n_estimators': [300, best_model.n_estimators],
    'max_depth': ['None', best_model.max_depth],
    'ROC-AUC': [baseline_result['ROC-AUC'], roc_auc],
    'PR-AUC': [baseline_result['PR-AUC'], pr_auc],
    'F1': [baseline_result['F1'], f1],
    'Accuracy': [baseline_result['Accuracy'], accuracy],
})

print('\nComparison: Baseline vs Optimized')
print('=' * 100)
print(comparison.to_string(index=False))

feature_reduction = (1 - len(optimal_cols) / int(baseline_result['Features'])) * 100
roc_auc_change = (roc_auc - baseline_result['ROC-AUC']) / baseline_result['ROC-AUC'] * 100
estimator_reduction = (1 - best_model.n_estimators / 300) * 100

print(f'\nOptimization Summary:')
print(f'  Feature reduction: {feature_reduction:.1f}%')
print(f'  Estimator reduction: {estimator_reduction:.1f}%')
print(f'  ROC-AUC change: {roc_auc_change:+.2f}%')
print(f'  Total parameter reduction: ~{(feature_reduction + estimator_reduction)/2:.0f}%')

In [ ]:
# Save Optimized Model

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
model_path = MODEL_DIR / f'rf_optimized_{timestamp}.pkl'
metadata_path = MODEL_DIR / f'rf_optimized_{timestamp}_metadata.json'
features_path = MODEL_DIR / f'rf_optimized_{timestamp}_features.json'

# Save model
joblib.dump(best_model, model_path)

# Save metadata
metadata = {
    'timestamp': timestamp,
    'model_type': 'RandomForestClassifier',
    'n_estimators': best_model.n_estimators,
    'max_depth': best_model.max_depth,
    'min_samples_split': int(best_model.min_samples_split),
    'class_weight': 'balanced',
    'random_state': RANDOM_SEED,
    'features_count': len(optimal_cols),
    'metrics': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'roc_auc': float(roc_auc),
        'pr_auc': float(pr_auc),
        'balanced_accuracy': float(balanced_acc),
        'mcc': float(mcc),
    },
    'comparison': {
        'baseline_roc_auc': float(baseline_result['ROC-AUC']),
        'feature_reduction_percent': float(feature_reduction),
        'estimator_reduction_percent': float(estimator_reduction),
        'roc_auc_change_percent': float(roc_auc_change),
    }
}

with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

# Save feature names
feature_info = {
    'feature_names': optimal_cols,
    'feature_count': len(optimal_cols),
    'feature_groups': {
        'C': [c for c in optimal_cols if c.startswith('C')],
        'D': [c for c in optimal_cols if c.startswith('D')],
        'M': [c for c in optimal_cols if c.startswith('M')],
        'engineered': [c for c in optimal_cols if c in ('uid', 'uid2')],
    }
}

with open(features_path, 'w') as f:
    json.dump(feature_info, f, indent=2)

print('Model Artifacts Saved:')
print(f'  Model: {model_path.name}')
print(f'  Metadata: {metadata_path.name}')
print(f'  Features: {features_path.name}')

## Summary

This notebook ranks the **eight models × four feature tables** written by `02_ml_models_fit.ipynb`. The compact Random Forest refit uses the reduced feature-engineering table (Table 3 filters plus `uid` / `uid2`).
